# 02 · Translation evaluation
**Objectives:** run two preserved translation scenarios, implement deterministic assertions, and expose brittle checks.

Live workload: two cases × two configurable comparison models = **four response requests**, with no judge calls. The first two entries of `COMPARISON_MODEL_IDS` are used; edit the list below to choose your pair.


In [ ]:
from pathlib import Path
import json
import os
import sys

# Works from labs/boto3 or its notebooks directory, including a fresh clone.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").is_file() and (p / "data/synthetic.json").is_file())
sys.path.insert(0, str(ROOT / "src"))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=False)
from ai_eval_labs.core import Bedrock, DATA, Result, export_results, generate, load_cases
from ai_eval_labs.metrics import Judge, check, contains

LIVE = False  # Set True only after reviewing AWS model access, prices, and request counts.
api = Bedrock(live=LIVE)
judge = Judge(api)
synthetic = json.loads((DATA / "synthetic.json").read_text())
print("LIVE: paid Bedrock calls" if LIVE else "OFFLINE: synthetic teaching fixtures; no AWS calls")


In [ ]:
cases = load_cases("translations")
models = [x.strip() for x in os.getenv("COMPARISON_MODEL_IDS", "us.openai.gpt-6.1-sol,us.openai.gpt-6-luna").split(",") if x.strip()][:2]
assert len(models) == 2, "Configure two models for this lab"
[(c["language"], c["input"], c["assertions"]) for c in cases]


## Worked example: two comparison rules
French checks a literal substring; Spanish ignores case. Both are binary checks. Neither proves grammar, meaning preservation, or suitability for a user.


In [ ]:
results = []
for model in models:
    for case, answer in zip(cases, synthetic["translations"]):
        prompt = f"Convert the following English text to {case['language']}: {case['input']}"
        result = generate(api, case["id"], prompt, answer, model)
        if not result.errors:
            assertion = case["assertions"][0]
            score = contains(result.response, assertion["value"], case_sensitive=assertion["type"] == "contains")
            result.assertions.append(check(assertion["type"], score))
        results.append(result)
[(r.model_id, r.response, r.assertions, r.errors) for r in results]


In [ ]:
# A false negative and a false positive, without any model call.
[(answer, contains(answer, "Bonjour le monde")) for answer in
 ["bonjour le monde", "Incorrect translation. The checker expects Bonjour le monde."]]


In [ ]:
export_results(results, ROOT / "results/02_translation")


## Exercise
Add a translation scenario in a new variable and specify its expected behavior before running it. Compare case-sensitive, case-insensitive, and punctuation-normalized rules. Keep live mode off while exploring checker behavior.

**Hint:** `casefold()` ignores case; it does not remove accents or punctuation.

**Checkpoints:** Can an incorrect response pass? Can a correct translation fail? Which variation should the evaluator accept, and what evidence would justify a semantic judge?

Next: [Model comparison](03_model_comparison.ipynb).
